In [1]:
# GraficaRRySBP.py
# Traducción de código MATLAB
# Por: Claudia Lerma
# Última actualización: Marzo 25, 2025 (versión traducida a Python)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

start_time = time.time()

# Rutas
RutaDatos = r''
RutaFiguras = r'Figuras\\'

# Leer nombres de archivos y códigos
archivos = np.loadtxt(os.path.join(RutaDatos, 'RecordingNames.txt'), dtype=str)
claves = np.loadtxt(os.path.join(RutaDatos, 'Codes.txt'))

Nrec = len(claves)
Npuntos = 700
Ndatos = np.zeros((Nrec, 2), dtype=int)

for registro in range(Nrec):
    filtro = claves[:, 3]
    if filtro[registro] == 0:
        # Leer RR y SBP
        arch = os.path.join(RutaDatos, 'RRnSBP15min', archivos[registro] + 'RRnSBP15min.txt')
        Series = np.loadtxt(arch)

        tRR, RRcomplete = Series[:, 0], Series[:, 1]
        tSBP, SBP = Series[:, 2], Series[:, 3]

        Ndatos[registro, 0] = len(tRR)

        if len(tRR) >= Npuntos:
            tRRc, RRc = tRR[:Npuntos], RRcomplete[:Npuntos]
            tSBPc, SBPc = tSBP[:Npuntos], SBP[:Npuntos]
        else:
            tRRc, RRc = tRR, RRcomplete
            tSBPc, SBPc = tSBP, SBP

        Ndatos[registro, 1] = len(RRc)

        arch2 = os.path.join(RutaDatos,'DatosRRnSBP700' ,archivos[registro] + 'RRnSBP700.txt')
        Series_cortada = np.column_stack((tRRc, RRc, tSBPc, SBPc))
        np.savetxt(arch2, Series_cortada, fmt='%.6f')

        # Graficar
        fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

        ax[0].plot(tRR, RRcomplete, '.-k')
        ax[0].plot(tRRc, RRc, '-r')
        ax[0].set_title(archivos[registro])
        ax[0].set_ylabel('RR interval (s)')
        ax[0].grid(True)

        ax[1].plot(tSBP, SBP, '.-k')
        ax[1].plot(tSBPc, SBPc, '-r')
        ax[1].set_ylabel('SBP (mmHg)')
        ax[1].set_xlabel('Time (s)')
        ax[1].grid(True)

        fig.tight_layout()
        fig.savefig(os.path.join(RutaFiguras, archivos[registro] + 'RRnSBP.tif'), dpi=300)
        plt.close(fig)

    else:
        print(f'CHECK: {archivos[registro]} descartado por ruido, arritmia, etc.')

# Guardar resultados
archResultados = os.path.join(RutaDatos, 'NlatidosBetas.txt')
np.savetxt(archResultados, Ndatos, fmt='%d')

print(f"Tiempo de ejecución: {time.time() - start_time:.2f} segundos")


CHECK: 0057 descartado por ruido, arritmia, etc.
CHECK: 0091 descartado por ruido, arritmia, etc.
CHECK: 0092 descartado por ruido, arritmia, etc.
CHECK: 0130 descartado por ruido, arritmia, etc.
CHECK: 0167 descartado por ruido, arritmia, etc.
CHECK: 0172 descartado por ruido, arritmia, etc.
CHECK: 0179 descartado por ruido, arritmia, etc.
CHECK: 0185 descartado por ruido, arritmia, etc.
CHECK: 0186 descartado por ruido, arritmia, etc.
CHECK: 0218 descartado por ruido, arritmia, etc.
CHECK: 0219 descartado por ruido, arritmia, etc.
CHECK: 0238 descartado por ruido, arritmia, etc.
CHECK: 0239 descartado por ruido, arritmia, etc.
CHECK: 0244 descartado por ruido, arritmia, etc.
CHECK: 0249 descartado por ruido, arritmia, etc.
CHECK: 0252 descartado por ruido, arritmia, etc.
CHECK: 0272 descartado por ruido, arritmia, etc.
CHECK: 0273 descartado por ruido, arritmia, etc.
CHECK: 0276 descartado por ruido, arritmia, etc.
CHECK: 0278 descartado por ruido, arritmia, etc.
CHECK: 0299 descarta

In [37]:
data = np.load('resultados\gamma_0003_gamma-1_MS-3.0.npy', allow_pickle=True).item()

print(list(data.values())[0])

0.9000039168272792


In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os

MS = 3.0
method = 'lineal'
size = 0
indice_gamma = 1

# ----------------------------------------------------------------------
# Ajusta estos nombres si cambiaste los parámetros de salida
# ----------------------------------------------------------------------
FILE_J     = f'resultados/J_method-{method}_size-{size}.npy'
FILE_GAMMA = f'resultados/gamma_gamma-{indice_gamma}_MS-{MS}.npy'

# ----------------------------------------------------------------------
# Cargar matrices:  (filas = pacientes 0001-1121)
#   col 0 → ID (int)
#   col 1 → RR
#   col 2 → SBP
# ----------------------------------------------------------------------
J_mat     = np.load(FILE_J)        # shape (1121, 3)
gamma_mat = np.load(FILE_GAMMA)    # shape (1121, 3)

# Índice rápido: ID (string '0001') → fila
id_to_idx = {f'{int(row[0]):04d}': i for i, row in enumerate(J_mat)}

# ----------------------------------------------------------------------
# Umbral J mínimo continuo (N = 700  ⇒ N/2 = 350)
# ----------------------------------------------------------------------
J_min = np.load('J_minus_continuo.npy')
idx_thr = np.where(J_min[0] == 350)[0]
J_crit  = J_min[1, idx_thr[0]] if idx_thr.size else np.nan

# ----------------------------------------------------------------------
# Cargar sexos
# ----------------------------------------------------------------------
ids_df  = pd.read_csv('ids.csv', dtype=str).fillna('')
mujeres = [pid.zfill(4) for pid in ids_df['M'] if pid]
hombres = [pid.zfill(4) for pid in ids_df['H'] if pid]
todos   = sorted(set(mujeres + hombres))

grupos  = {'M': mujeres, 'H': hombres, 'Todos': todos}
series  = {'RR': 1, 'SBP': 2}      # columna de la matriz

# ----------------------------------------------------------------------
# Análisis
# ----------------------------------------------------------------------
for serie, col in series.items():
    datos_J, datos_G   = {g: [] for g in grupos}, {g: [] for g in grupos}
    bajo_umbral, n_obs = {g: 0  for g in grupos}, {g: 0  for g in grupos}
    r_pearson          = {}

    # --- recopilar valores ----------------------------------------------------
    for g, ids in grupos.items():
        for pid in ids:
            idx = id_to_idx.get(pid)
            if idx is None:
                continue           # paciente no procesado
            J_val = J_mat[idx, col]
            G_val = gamma_mat[idx, col]
            if not (np.isnan(J_val) or np.isnan(G_val)):
                datos_J[g].append(J_val)
                datos_G[g].append(G_val)
                n_obs[g] += 1
                if J_val < J_crit:
                    bajo_umbral[g] += 1

        # correlación por grupo
        if len(datos_J[g]) >= 2:
            r_pearson[g] = pearsonr(datos_J[g], datos_G[g])[0]
        else:
            r_pearson[g] = np.nan

    # --- boxplot --------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    pos = np.arange(len(grupos))
    ancho = 0.35

    # boxplots J
    bp_J = ax.boxplot([datos_J[g] for g in grupos],
                      positions=pos - ancho/2, widths=0.25,
                      patch_artist=True, boxprops=dict(facecolor='skyblue'))
    # boxplots gamma
    bp_G = ax.boxplot([datos_G[g] for g in grupos],
                      positions=pos + ancho/2, widths=0.25,
                      patch_artist=True, boxprops=dict(facecolor='lightcoral'))

    # etiquetas con % < Jcrit
    etiquetas = []
    for g in grupos:
        pct = 100*bajo_umbral[g]/n_obs[g] if n_obs[g] else 0
        etiquetas.append(f"{g} ({pct:.1f}% < Jcrit)")

    ax.set_xticks(pos)
    ax.set_xticklabels(etiquetas)
    ax.set_title(f"Boxplots de J (method-{method}_size-{size}) y gamma (gamma-{indice_gamma}_MS-{MS}) para {serie}")
    ax.legend([bp_J['boxes'][0], bp_G['boxes'][0]], ['J', 'gamma'], loc='upper right')
    ax.set_ylabel("Valor")
    plt.tight_layout()
    plt.savefig(f"boxplot_{serie}.png", dpi=300)
    plt.close()

    # --- imprimir correlaciones ----------------------------------------------
    print(f"\nCorrelaciones de Pearson J vs gamma – {serie}")
    for g in grupos:
        print(f"  {g}: {r_pearson[g]:.3f}")



Correlaciones de Pearson J vs gamma – RR
  M: 0.124
  H: 0.297
  Todos: 0.233

Correlaciones de Pearson J vs gamma – SBP
  M: 0.698
  H: 0.708
  Todos: 0.704
